# Notebook 2: Variance Reduction

Naive Monte Carlo converges at $O(1/\sqrt{N})$ — doubling precision costs four times
as many paths. This notebook measures that cost directly, then implements two standard
variance reduction techniques — antithetic variates and control variates — in pure NumPy.
The goal is to see exactly where the improvement comes from and how large it is in
practice, without relying on the C++ engine's internals.

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(),
    '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
sys.path.insert(0, os.path.join(REPO_ROOT, 'build'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import norm
import math

# Common Black-Scholes helper
def bs_call(S, K, r, sigma, T):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

# Option parameters
S0, K, r, sigma, T = 100.0, 105.0, 0.05, 0.20, 1.0
analytical = bs_call(S0, K, r, sigma, T)
print(f'Analytical price: {analytical:.4f}')

## Section 1 — Naive Monte Carlo Convergence

Under the risk-neutral measure, the terminal stock price is:

$$S_T = S_0 \exp\!\left[(r - \tfrac{1}{2}\sigma^2)T + \sigma\sqrt{T}\,Z\right], \quad Z\sim\mathcal{N}(0,1)$$

The discounted payoff estimator has standard deviation $\approx \sigma_\text{payoff}/\sqrt{N}$,
so the standard deviation of the **price estimate** shrinks as $1/\sqrt{N}$.  
We verify this empirically by running 10 independent trials at each path count.

In [ ]:
rng = np.random.default_rng(0)

def mc_call_naive(n, seed):
    """Price European call via direct GBM simulation (pure NumPy)."""
    rng_l = np.random.default_rng(seed)
    Z   = rng_l.standard_normal(n)
    ST  = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    pay = np.maximum(ST - K, 0.0)
    return np.exp(-r * T) * np.mean(pay)

path_counts = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000, 1_000_000]
n_trials    = 10
naive_stds  = []

for n in path_counts:
    prices = [mc_call_naive(n, seed=i) for i in range(n_trials)]
    std = np.std(prices, ddof=1)
    naive_stds.append(std)
    print(f'  N={n:>9,}: std={std:.5f}')

ns  = np.array(path_counts, dtype=float)
C_naive = naive_stds[0] * np.sqrt(path_counts[0])

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(ns, naive_stds, 'bo-', linewidth=2, markersize=6, label='Naive MC std')
ax.loglog(ns, C_naive / np.sqrt(ns), 'b--', linewidth=1.5, label=r'$C/\sqrt{N}$ reference')
ax.set_xlabel('Number of Paths', fontsize=13)
ax.set_ylabel('Std of Price Estimate  (10 trials)', fontsize=13)
ax.set_title('Naive MC: Standard Deviation of Estimate vs Path Count', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
plt.show()

## Section 2 — Antithetic Variates

**Idea**: for every standard normal draw $Z$, also simulate the *mirror* path with $-Z$.
Average the two payoffs:

$$\hat{C}_{\text{anti}} = e^{-rT} \frac{1}{N/2} \sum_{i=1}^{N/2} \frac{f(Z_i) + f(-Z_i)}{2}$$

Because $f(Z)$ and $f(-Z)$ are negatively correlated (when the stock goes up, the
mirror path goes down), the variance of the average is reduced:

$$\text{Var}\!\left[\tfrac{f(Z)+f(-Z)}{2}\right] = \tfrac{1}{2}\text{Var}[f] + \tfrac{1}{2}\text{Cov}[f(Z),f(-Z)] < \text{Var}[f]$$

The computational cost per path is essentially unchanged.

In [ ]:
def mc_call_antithetic(n, seed):
    """Price European call using antithetic variates (pure NumPy)."""
    assert n % 2 == 0, 'n must be even for antithetic variates'
    rng_l = np.random.default_rng(seed)
    Z  = rng_l.standard_normal(n // 2)
    # Two paths: +Z and -Z
    ST_pos = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) *  Z)
    ST_neg = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * -Z)
    pay = 0.5 * (np.maximum(ST_pos - K, 0.0) + np.maximum(ST_neg - K, 0.0))
    return np.exp(-r * T) * np.mean(pay)

anti_stds = []
for n in path_counts:
    prices = [mc_call_antithetic(n, seed=i) for i in range(n_trials)]
    std = np.std(prices, ddof=1)
    anti_stds.append(std)

# Variance reduction factor at 100K paths
idx_100k = path_counts.index(100_000)
var_reduction_anti = (naive_stds[idx_100k] / anti_stds[idx_100k]) ** 2
print(f'Variance reduction factor (antithetic, N=100K): {var_reduction_anti:.2f}×')

C_anti = anti_stds[0] * np.sqrt(path_counts[0])

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(ns, naive_stds,  'bo-', linewidth=2, markersize=6, label='Naive')
ax.loglog(ns, anti_stds,   'rs-', linewidth=2, markersize=6, label='Antithetic variates')
ax.loglog(ns, C_naive / np.sqrt(ns), 'b--', linewidth=1, alpha=0.5)
ax.loglog(ns, C_anti  / np.sqrt(ns), 'r--', linewidth=1, alpha=0.5)
ax.set_xlabel('Number of Paths', fontsize=13)
ax.set_ylabel('Std of Price Estimate  (10 trials)', fontsize=13)
ax.set_title('Naive vs Antithetic Variates: Convergence', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
plt.show()

## Section 3 — Control Variates

**Idea**: use a *correlated* variable with a **known** expected value to reduce variance.

Choose $X = S_T$ (the terminal stock price).  Under the risk-neutral measure:

$$E^Q[S_T] = S_0 e^{rT}$$

The control variate estimator is:

$$\hat{C}_{\text{CV}} = e^{-rT} \left[\frac{1}{N}\sum_i f(S_T^{(i)})
  - c\left(\frac{1}{N}\sum_i S_T^{(i)} - S_0 e^{rT}\right)\right]$$

where the optimal coefficient is:

$$c^* = \frac{\text{Cov}(f(S_T),\, S_T)}{\text{Var}(S_T)}$$

estimated from the same sample batch.  The variance reduction is proportional to
$1 - \rho^2$ where $\rho$ is the correlation between payoff and $S_T$.

In [ ]:
def mc_call_cv(n, seed):
    """Price European call using S_T as control variate (pure NumPy)."""
    rng_l = np.random.default_rng(seed)
    Z     = rng_l.standard_normal(n)
    ST    = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    pay   = np.maximum(ST - K, 0.0)
    mu_ST = S0 * np.exp(r * T)          # known E[S_T]
    # Estimate optimal c from this batch
    cov_fp = np.cov(pay, ST)[0, 1]
    var_ST = np.var(ST, ddof=1)
    c      = cov_fp / var_ST if var_ST > 1e-30 else 0.0
    pay_cv = pay - c * (ST - mu_ST)
    return np.exp(-r * T) * np.mean(pay_cv)

cv_stds = []
for n in path_counts:
    prices = [mc_call_cv(n, seed=i) for i in range(n_trials)]
    std = np.std(prices, ddof=1)
    cv_stds.append(std)

var_reduction_cv = (naive_stds[idx_100k] / cv_stds[idx_100k]) ** 2
print(f'Variance reduction factor (control variates, N=100K): {var_reduction_cv:.2f}×')

C_cv = cv_stds[0] * np.sqrt(path_counts[0])

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(ns, naive_stds, 'bo-', linewidth=2, markersize=6, label='Naive')
ax.loglog(ns, anti_stds,  'rs-', linewidth=2, markersize=6, label='Antithetic variates')
ax.loglog(ns, cv_stds,    'g^-', linewidth=2, markersize=6, label='Control variates ($S_T$)')
ax.loglog(ns, C_naive / np.sqrt(ns), 'b--', linewidth=1, alpha=0.4)
ax.loglog(ns, C_anti  / np.sqrt(ns), 'r--', linewidth=1, alpha=0.4)
ax.loglog(ns, C_cv    / np.sqrt(ns), 'g--', linewidth=1, alpha=0.4)
ax.set_xlabel('Number of Paths', fontsize=13)
ax.set_ylabel('Std of Price Estimate  (10 trials)', fontsize=13)
ax.set_title('Variance Reduction Comparison', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
plt.show()

## Section 4 — Summary Table

The **equivalent speedup** is the factor by which fewer paths you need to reach the
same variance as naive MC: if variance drops by $k\times$, you only need $N/k$ paths.

In [ ]:
var_naive = naive_stds[idx_100k] ** 2
var_anti  = anti_stds[idx_100k]  ** 2
var_cv    = cv_stds[idx_100k]    ** 2

rows = [
    ('Naive MC',           var_naive, 1.0),
    ('Antithetic variates', var_anti,  var_naive / var_anti),
    ('Control variates',   var_cv,    var_naive / var_cv),
]

hdr = f"{'Method':<24} | {'Variance @100K':>14} | {'VR Factor':>10} | {'Equiv Speedup':>14}"
sep = '-' * len(hdr)
print(sep)
print(hdr)
print(sep)
for name, var, vr in rows:
    print(f"{name:<24} | {var:>14.6f} | {vr:>10.2f}× | {vr:>13.1f}×")
print(sep)
print()
print('Variance Reduction Factor = Var(naive) / Var(method)')
print('Equivalent Speedup       = same factor (fewer paths for same variance)')